In [1]:
!git clone https://github.com/boracandan/Algoverse_Truth_Directions_Research.git

Cloning into 'Algoverse_Truth_Directions_Research'...
remote: Enumerating objects: 496, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 496 (delta 30), reused 61 (delta 23), pack-reused 413 (from 1)
Receiving objects: 100% (496/496), 102.64 MiB | 30.50 MiB/s, done.
Resolving deltas: 100% (251/251), done.
Updating files: 100% (172/172), done.


In [2]:
from ast import literal_eval
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import pickle
import gc
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import roc_auc_score

print("imports complete")

imports complete


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

csv_path = "/content/drive/MyDrive/algoverse_results/results_database.csv"  # or the full path if not in the same directory

# Create empty DataFrame with just headers
df_empty = pd.DataFrame(columns=[
    'train_task', 'test_task', 'train_condition', 'test_condition', 
    'layer', 'model', 'auroc'
])

df_empty.to_csv(csv_path, index=False)

print("✓ CSV cleared completely")
print(f"File: {csv_path}")
print(f"Rows: 0 (headers only)")

✓ CSV cleared completely
File: /content/drive/MyDrive/algoverse_results/results_database.csv
Rows: 0 (headers only)


In [5]:
save_dir = "/content/drive/MyDrive/algoverse_results"
DATA_DIR = "/content/Algoverse_Truth_Directions_Research/datasets/CoT_datasets/lexically_cleaned"
os.makedirs(save_dir, exist_ok=True)
csv_path = f"{save_dir}/results_database.csv"

In [6]:
gc.collect()
torch.cuda.empty_cache()

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"


model = AutoModelForCausalLM.from_pretrained(
    "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True,
)
model.eval()

print(f"✓ Model loaded")
print(f"  Device: {model.device}")
print(f"  GPU memory: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
print(f"  Layers: {model.config.num_hidden_layers}")

Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✓ Model loaded
  Device: cuda:0
  GPU memory: 16.06 GB
  Layers: 32


In [7]:
print(f"Loading lexically_cleaned (cot-zero-shot) datasets from {DATA_DIR}...")

# Load all task datasets
F0_train = pd.read_csv(f"{DATA_DIR}/F0_train.csv")
F0_test = pd.read_csv(f"{DATA_DIR}/F0_test.csv")
F1_train = pd.read_csv(f"{DATA_DIR}/F1_train.csv")
F1_test = pd.read_csv(f"{DATA_DIR}/F1_test.csv")
F2_train = pd.read_csv(f"{DATA_DIR}/F2_train.csv")
F2_test = pd.read_csv(f"{DATA_DIR}/F2_test.csv")
F3_train = pd.read_csv(f"{DATA_DIR}/F3_train.csv")
F3_test = pd.read_csv(f"{DATA_DIR}/F3_test.csv")
F4_train = pd.read_csv(f"{DATA_DIR}/F4_train.csv")
F4_test = pd.read_csv(f"{DATA_DIR}/F4_test.csv")
F5_train = pd.read_csv(f"{DATA_DIR}/F5_train.csv")
F5_test = pd.read_csv(f"{DATA_DIR}/F5_test.csv")
A1_train = pd.read_csv(f"{DATA_DIR}/A1_train.csv")
A1_test = pd.read_csv(f"{DATA_DIR}/A1_test.csv")
A2_train = pd.read_csv(f"{DATA_DIR}/A2_train.csv")
A2_test = pd.read_csv(f"{DATA_DIR}/A2_test.csv")
A3_train = pd.read_csv(f"{DATA_DIR}/A3_train.csv")
A3_test = pd.read_csv(f"{DATA_DIR}/A3_test.csv")

tasks_dict = {
    "F0": (F0_train, F0_test), "F1": (F1_train, F1_test), "F2": (F2_train, F2_test),
    "F3": (F3_train, F3_test), "F4": (F4_train, F4_test), "F5": (F5_train, F5_test),
    "A1": (A1_train, A1_test), "A2": (A2_train, A2_test), "A3": (A3_train, A3_test),
}
task_names = ["A1", "A2", "A3", "F0", "F1", "F2", "F3", "F4", "F5"]

print("✓ All lexically_cleaned (cot-zero-shot) datasets loaded:")
for name, (tr, te) in tasks_dict.items():
    print(f"    {name}: {len(tr)} train / {len(te)} test")

Loading lexically_cleaned (cot-zero-shot) datasets from /content/Algoverse_Truth_Directions_Research/datasets/CoT_datasets/lexically_cleaned...
✓ All lexically_cleaned (cot-zero-shot) datasets loaded:
    F0: 796 train / 348 test
    F1: 972 train / 426 test
    F2: 494 train / 202 test
    F3: 1088 train / 478 test
    F4: 984 train / 458 test
    F5: 670 train / 292 test
    A1: 282 train / 210 test
    A2: 272 train / 112 test
    A3: 298 train / 130 test


In [8]:
def decode_ids(id_column):
    return [
        tokenizer.decode(literal_eval(ids) if isinstance(ids, str) else ids)
        for ids in id_column
    ]


def activations_all_layers(model, statements, batch_size=4):
    statements = list(statements)
    num_layers = model.config.num_hidden_layers + 1
    activations_by_layer = [[] for _ in range(num_layers)]

    for i in range(0, len(statements), batch_size):
        batch = statements[i : i + batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)

        for layer_idx, layer_hidden in enumerate(outputs.hidden_states):
            final_token = layer_hidden[:, -1, :]
            activations_by_layer[layer_idx].append(final_token.cpu())

        del outputs, inputs
        torch.cuda.empty_cache()

    return [torch.cat(layer_acts, dim=0) for layer_acts in activations_by_layer]


def train_probe(activations, labels, device='cuda'):
    (X_train, X_test), (y_train, y_test) = activations, labels
    X_train = torch.stack(list(X_train)).float().numpy()
    X_test = torch.stack(list(X_test)).float().numpy()
    y_train = y_train.to_numpy()
    y_test = y_test.to_numpy()

    train_mean = X_train.mean(axis=0)
    X_train = X_train - train_mean
    X_test = X_test - train_mean

    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)

    hidden_dim = X_train.shape[1]
    probe = nn.Linear(hidden_dim, 1, bias=False).to(device)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-3, weight_decay=0.1)
    loss_fn = nn.BCEWithLogitsLoss()

    for _ in range(1000):
        optimizer.zero_grad()
        loss = loss_fn(probe(X_train_t).squeeze(-1), y_train_t)
        loss.backward()
        optimizer.step()

    probe.eval()
    with torch.no_grad():
        test_logits = probe(X_test_t).squeeze(-1).cpu().numpy()

    auroc = roc_auc_score(y_test, test_logits)
    w = probe.weight.detach().cpu().numpy().flatten()
    return w, train_mean, auroc


def train_all_layers(train_acts, test_acts, y_train, y_test):
    num_layers = len(train_acts)
    layer_results = {}
    for layer_idx in range(num_layers):
        X_train = train_acts[layer_idx]
        X_test = test_acts[layer_idx]
        X_train_np = torch.stack(list(X_train)).float().numpy()
        variance = X_train_np.var(axis=0) + 1e-6
        w, train_mean, auroc = train_probe((X_train, X_test), (y_train, y_test))
        layer_results[layer_idx] = {
            "auroc": auroc,
            "weights": w,
            "variance": variance,
            "train_mean": train_mean,
        }
        if layer_idx % 8 == 0:
            print(f"    Layer {layer_idx}: AUROC = {auroc:.4f}")
    return layer_results


def write_indomain_rows(task_name, layer_results, csv_path, condition="cot-zero-shot"):
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        mask = ~((df["train_task"] == task_name) & 
                 (df["test_task"] == task_name) & 
                 (df["train_condition"] == condition) &
                 (df["model"] == "deepseek-r1-distill-8b"))
        df = df[mask]
    else:
        df = pd.DataFrame(columns=["train_task","test_task","train_condition","test_condition","layer","model","auroc"])

    new_rows = []
    for layer_idx, data in layer_results.items():
        new_rows.append({
            "train_task": task_name,
            "test_task": task_name,
            "train_condition": condition,
            "test_condition": condition,
            "layer": layer_idx,
            "model": "deepseek-r1-distill-8b",
            "auroc": data["auroc"],
        })

    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print(f"  ✓ Saved {len(new_rows)} layers for {task_name} ({condition})")

print("✓ All functions defined")

✓ All functions defined


In [9]:

task_name = "A1"
print(f"\nProcessing {task_name} (cot-zero-shot)...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_A1_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_A1_cot, csv_path, condition="cot-zero-shot")


Processing A1 (cot-zero-shot)...
    Layer 0: AUROC = 0.3997
    Layer 8: AUROC = 0.9809
    Layer 16: AUROC = 0.9915
    Layer 24: AUROC = 0.9906
    Layer 32: AUROC = 0.9903
  ✓ Saved 33 layers for A1 (cot-zero-shot)


/tmp/ipykernel_1533/2141827977.py:108: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)


In [10]:

task_name = "A2"
print(f"\nProcessing {task_name} (cot-zero-shot)...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_A2_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_A2_cot, csv_path, condition="cot-zero-shot")


Processing A2 (cot-zero-shot)...
    Layer 0: AUROC = 0.5776
    Layer 8: AUROC = 0.9621
    Layer 16: AUROC = 1.0000
    Layer 24: AUROC = 1.0000
    Layer 32: AUROC = 0.9997
  ✓ Saved 33 layers for A2 (cot-zero-shot)


In [11]:

task_name = "A3"
print(f"\nProcessing {task_name} (cot-zero-shot)...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_A3_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_A3_cot, csv_path, condition="cot-zero-shot")


Processing A3 (cot-zero-shot)...
    Layer 0: AUROC = 0.5338
    Layer 8: AUROC = 0.9870
    Layer 16: AUROC = 1.0000
    Layer 24: AUROC = 1.0000
    Layer 32: AUROC = 1.0000
  ✓ Saved 33 layers for A3 (cot-zero-shot)


In [12]:
task_name = "F0"
print(f"\nProcessing {task_name} (cot-zero-shot)...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F0_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F0_cot, csv_path, condition="cot-zero-shot")


Processing F0 (cot-zero-shot)...
    Layer 0: AUROC = 0.5000
    Layer 8: AUROC = 0.8490
    Layer 16: AUROC = 0.9696
    Layer 24: AUROC = 0.9779
    Layer 32: AUROC = 0.9840
  ✓ Saved 33 layers for F0 (cot-zero-shot)


In [13]:

task_name = "F1"
print(f"\nProcessing {task_name} (cot-zero-shot)...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F1_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F1_cot, csv_path, condition="cot-zero-shot")


Processing F1 (cot-zero-shot)...
    Layer 0: AUROC = 0.5000
    Layer 8: AUROC = 0.8886
    Layer 16: AUROC = 0.9895
    Layer 24: AUROC = 0.9882
    Layer 32: AUROC = 0.9877
  ✓ Saved 33 layers for F1 (cot-zero-shot)


In [14]:

task_name = "F2"
print(f"\nProcessing {task_name} (cot-zero-shot)...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F2_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F2_cot, csv_path, condition="cot-zero-shot")


Processing F2 (cot-zero-shot)...
    Layer 0: AUROC = 0.5000
    Layer 8: AUROC = 0.8707
    Layer 16: AUROC = 0.9561
    Layer 24: AUROC = 0.9544
    Layer 32: AUROC = 0.9479
  ✓ Saved 33 layers for F2 (cot-zero-shot)


In [15]:

task_name = "F3"
print(f"\nProcessing {task_name} (cot-zero-shot)...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F3_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F3_cot, csv_path, condition="cot-zero-shot")


Processing F3 (cot-zero-shot)...
    Layer 0: AUROC = 0.4965
    Layer 8: AUROC = 0.8968
    Layer 16: AUROC = 0.9721
    Layer 24: AUROC = 0.9765
    Layer 32: AUROC = 0.9780
  ✓ Saved 33 layers for F3 (cot-zero-shot)


In [16]:

task_name = "F4"
print(f"\nProcessing {task_name} (cot-zero-shot)...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F4_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F4_cot, csv_path, condition="cot-zero-shot")


Processing F4 (cot-zero-shot)...
    Layer 0: AUROC = 0.5000
    Layer 8: AUROC = 0.8550
    Layer 16: AUROC = 0.9660
    Layer 24: AUROC = 0.9636
    Layer 32: AUROC = 0.9570
  ✓ Saved 33 layers for F4 (cot-zero-shot)


In [17]:

task_name = "F5"
print(f"\nProcessing {task_name} (cot-zero-shot)...")
train_df, test_df = tasks_dict[task_name]
train_acts = activations_all_layers(model, decode_ids(train_df["extracted_statement_ids"]), batch_size=4)
test_acts = activations_all_layers(model, decode_ids(test_df["extracted_statement_ids"]), batch_size=4)
results_F5_cot = train_all_layers(train_acts, test_acts, train_df["label"], test_df["label"])
del train_acts, test_acts
torch.cuda.empty_cache()
write_indomain_rows(task_name, results_F5_cot, csv_path, condition="cot-zero-shot")


Processing F5 (cot-zero-shot)...


    Layer 0: AUROC = 0.5499
    Layer 8: AUROC = 0.8915
    Layer 16: AUROC = 0.9705
    Layer 24: AUROC = 0.9734
    Layer 32: AUROC = 0.9727
  ✓ Saved 33 layers for F5 (cot-zero-shot)


In [18]:
print("Pre-extracting FULL (train+test) activations for all tasks (cot-zero-shot)...")
full_activations = {}
full_labels = {}

for task_name in task_names:
    print(f"  Extracting {task_name}...")
    train_df, test_df = tasks_dict[task_name]
    full_df = pd.concat([train_df, test_df], ignore_index=True)  # use everything for out-of-domain testing
    full_labels[task_name] = full_df["label"].to_numpy()

    acts = activations_all_layers(model, decode_ids(full_df["extracted_statement_ids"]), batch_size=4)
    full_activations[task_name] = [
        layer_acts.float().numpy() if isinstance(layer_acts, torch.Tensor)
        else torch.stack(list(layer_acts)).float().numpy()
        for layer_acts in acts
    ]
    del acts
    torch.cuda.empty_cache()

print("\n All full activations extracted")


Pre-extracting FULL (train+test) activations for all tasks (cot-zero-shot)...
  Extracting A1...
  Extracting A2...
  Extracting A3...
  Extracting F0...
  Extracting F1...
  Extracting F2...
  Extracting F3...
  Extracting F4...
  Extracting F5...

 All full activations extracted


In [19]:
all_probes = {
    "A1": results_A1_cot, "A2": results_A2_cot, "A3": results_A3_cot,
    "F0": results_F0_cot, "F1": results_F1_cot, "F2": results_F2_cot,
    "F3": results_F3_cot, "F4": results_F4_cot, "F5": results_F5_cot,
}

df = pd.read_csv(csv_path)
mask = ~(
    (df["train_task"] != df["test_task"]) &
    (df["model"] == "deepseek-r1-distill-8b") &
    (df["train_condition"] == "cot-zero-shot")
)
df = df[mask]

num_layers = model.config.num_hidden_layers + 1
cross_task_rows = []
total = len(task_names) * (len(task_names) - 1) * num_layers
completed = 0

for train_task in task_names:
    for test_task in task_names:
        if train_task == test_task:
            continue

        # Out-of-domain: evaluate on the FULL test_task data (train+test combined), not
        # just its held-out test split -- there's no train/test leakage concern here since
        # the probe was never trained on test_task at all.
        test_labels = full_labels[test_task]

        for layer_idx in range(num_layers):
            w = all_probes[train_task][layer_idx]["weights"]
            mean_train = all_probes[train_task][layer_idx]["train_mean"]

            X_test = full_activations[test_task][layer_idx]
            X_test_centered = X_test - mean_train
            logits = X_test_centered @ w
            auroc = roc_auc_score(test_labels, logits)

            cross_task_rows.append({
                "train_task": train_task,
                "test_task": test_task,
                "train_condition": "cot-zero-shot",
                "test_condition": "cot-zero-shot",
                "layer": layer_idx,
                "model": "deepseek-r1-distill-8b",
                "auroc": auroc,
            })

            completed += 1
            if completed % 500 == 0:
                print(f"  {completed}/{total} done...")

df = pd.concat([df, pd.DataFrame(cross_task_rows)], ignore_index=True)
df.to_csv(csv_path, index=False)

print(f"\n✓ Done. Total rows: {len(df)}")
print(f"  In-domain: {len(df[df['train_task'] == df['test_task']])}")
print(f"  Cross-task: {len(df[df['train_task'] != df['test_task']])}")


  500/2376 done...
  1000/2376 done...
  1500/2376 done...
  2000/2376 done...

✓ Done. Total rows: 2673
  In-domain: 297
  Cross-task: 2376


In [20]:
import pandas as pd

csv_path = "/Users/Syam/truth_directions/Algoverse_Truth_Directions_Research/experiments/results_database.csv"
df = pd.read_csv(csv_path)

print("=" * 60)
print("CSV VALIDATION")
print("=" * 60)

print(f"\nTotal rows: {len(df)}")

print(f"\nRows by condition:")
conditions = df.groupby('train_condition').size()
print(conditions)

print(f"\nRows by model:")
models = df.groupby('model').size()
print(models)

print(f"\nIn-domain rows (train_task == test_task):")
indomain = df[df['train_task'] == df['test_task']]
print(f"Total: {len(indomain)}")
print(indomain.groupby('train_condition').size())

print(f"\nCross-task rows (train_task != test_task):")
crosstask = df[df['train_task'] != df['test_task']]
print(f"Total: {len(crosstask)}")
print(crosstask.groupby('train_condition').size())

print(f"\nLayer 25 AUROC by condition:")
layer25 = df[df['layer'] == 25].pivot_table(
    index='train_task',
    columns='train_condition',
    values='auroc'
)
print(layer25.round(4))

print(f"\n✓ Validation complete")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/Syam/truth_directions/Algoverse_Truth_Directions_Research/experiments/results_database.csv'